# MiniGrid MPC: H16/K8 vs H40/K20

This notebook aggregates five world-model seeds, five target maps, and ten episodes per seed/map. It validates all 1,000 episode rows in each configuration before displaying the four-baseline comparison with a two-level column header.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'wm').is_dir() and (candidate / 'trainer').is_dir():
            return candidate
    raise FileNotFoundError('Could not find repository root containing wm/ and trainer/')

REPO_ROOT = find_repo_root()
CONFIGS = {
    'H16 / K8': {
        'horizon': 16,
        'execute_steps': 8,
        'csv': REPO_ROOT / 'wm/outputs/planning/mpc_baseline_target_eval/mpc_episode_results.csv',
    },
    'H40 / K20': {
        'horizon': 40,
        'execute_steps': 20,
        'csv': REPO_ROOT / 'outputs/planning/mpc_baseline_target_eval_h40_k20/mpc_episode_results.csv',
    },
}
BASELINES = ['mac', 'target', 'dr', 'p2e']
TARGETS = [f'target_task{i}' for i in range(5)]
WM_SEEDS = list(range(5))
EPISODES_PER_CASE = 10
EXPECTED_EPISODES = len(BASELINES) * len(TARGETS) * len(WM_SEEDS) * EPISODES_PER_CASE

CONFIGS

{'H16 / K8': {'horizon': 16,
  'execute_steps': 8,
  'csv': PosixPath('/home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/wm/outputs/planning/mpc_baseline_target_eval/mpc_episode_results.csv')},
 'H40 / K20': {'horizon': 40,
  'execute_steps': 20,
  'csv': PosixPath('/home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/outputs/planning/mpc_baseline_target_eval_h40_k20/mpc_episode_results.csv')}}

In [2]:
def load_and_validate(label, spec):
    path = spec['csv']
    if not path.is_file():
        raise FileNotFoundError(f'{label}: missing {path}')
    frame = pd.read_csv(path)
    required = {
        'baseline', 'wm_seed', 'target', 'episode', 'success',
        'dense_return', 'environment_steps', 'imagined_transitions', 'execute_steps',
    }
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{label}: missing columns {sorted(missing)}')

    frame = frame[
        frame['baseline'].isin(BASELINES)
        & frame['target'].isin(TARGETS)
        & frame['wm_seed'].isin(WM_SEEDS)
    ].copy()
    identity = ['baseline', 'wm_seed', 'target', 'episode']
    if len(frame) != EXPECTED_EPISODES or frame.duplicated(identity).any():
        unique = len(frame.drop_duplicates(identity))
        raise ValueError(
            f'{label}: expected {EXPECTED_EPISODES} unique episodes; '
            f'found {len(frame)} rows and {unique} unique identities'
        )
    counts = frame.groupby(['baseline', 'wm_seed', 'target']).size()
    if not counts.eq(EPISODES_PER_CASE).all():
        raise ValueError(f'{label}: at least one case does not contain 10 episodes')
    if not frame['execute_steps'].eq(spec['execute_steps']).all():
        raise ValueError(f"{label}: execute_steps does not equal {spec['execute_steps']}")
    if 'horizon' in frame and not frame['horizon'].eq(spec['horizon']).all():
        raise ValueError(f"{label}: horizon does not equal {spec['horizon']}")
    frame['success'] = frame['success'].astype(str).str.lower().eq('true')
    return frame

episode_data = {label: load_and_validate(label, spec) for label, spec in CONFIGS.items()}
pd.DataFrame({
    label: {'episode_rows': len(frame), 'unique_cases': frame.groupby(['baseline', 'wm_seed', 'target']).ngroups}
    for label, frame in episode_data.items()
}).T

,episode_rows,unique_cases
H16 / K8,1000,100
H40 / K20,1000,100


In [3]:
metric_names = ['Success rate', 'Dense reward', 'Env steps', 'Imagined steps']
configuration_tables = {}
for label, frame in episode_data.items():
    aggregated = frame.groupby('baseline', sort=False).agg(
        **{
            'Success rate': ('success', 'mean'),
            'Dense reward': ('dense_return', 'mean'),
            'Env steps': ('environment_steps', 'mean'),
            'Imagined steps': ('imagined_transitions', 'mean'),
        }
    ).reindex(BASELINES)
    configuration_tables[label] = aggregated[metric_names]

comparison = pd.concat(configuration_tables, axis=1)
comparison.index = ['MAC', 'Target', 'DR', 'P2E']
comparison.index.name = 'Baseline'

formatters = {}
for config in CONFIGS:
    formatters[(config, 'Success rate')] = '{:.1%}'
    formatters[(config, 'Dense reward')] = '{:.3f}'
    formatters[(config, 'Env steps')] = '{:.2f}'
    formatters[(config, 'Imagined steps')] = '{:,.0f}'

styled_comparison = (
    comparison.style
    .format(formatters)
    .set_caption('Five targets × five WM seeds × ten episodes per case')
    .set_properties(**{'text-align': 'right'})
)
display(styled_comparison)

In [4]:
output_path = REPO_ROOT / 'outputs/planning/mpc_h16k8_vs_h40k20_pivot.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(output_path)  # Keeps the two header rows.
print(f'Saved: {output_path}')

Saved: /home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/outputs/planning/mpc_h16k8_vs_h40k20_pivot.csv
